# Lab 3 (Chapter 4, Part I) (optional): Logistic Regression and Classification Metrics

**CS 463 · Optional practice, not for credit · Nothing to submit**

There's no lab period this week, so this lab is optional practice for Chapter 4. Work through it whenever it suits you. If you don't have much time, I'd highly recommend that you at least read through it. Every exercise comes with a solution you can open, but you'll get much more out of it if you try each one before you look.

### References:

The lab occasionally refers to concepts covered in these two notebooks:

  - **NB1:** `ch_4_demos_1_log_odds_and_logistic_regression.ipynb`
  - **NB2:** `ch_4_demos_2_classification_metrics.ipynb`

  Each exercise says which section to look back at.
- **Real tools.** You'll use scikit-learn, pandas, and SciPy, the same libraries you'd use outside this class. The demos hid their code in a helper file; here you write it yourself.
- **Hints** are folded up under the prompts. For most code exercises, **Hint 1** is a code skeleton: paste it into the cell and replace each `...`. It shows what to compute and which names later cells need, but not how. **Hint 2**, when there is one, points toward a method. Open them when you're stuck.
- **Later cells use earlier results.** Each exercise says which variables it creates (for example, `bal_model`). If you skip an exercise or get stuck, copy its solution into the cell so later cells keep working.
- **Code** exercises go in the cell right below the prompt. **Written** exercises go in the **Answer:** cell below. Written answers: 1–3 sentences.
- **Solutions** are folded up right below each code cell (**Solution**) and each answer cell (**Our answer**). Try the exercise first, then open them to check your work.
- **Priority** If you're pressed for time, you can skim or skip things I've marked "(Low Priority)". Pay particular attention to things I marked "(High Priority)".

**Checklist**

| Part | Topic | Exercises |
|---|---|---|
| 1 | A model with one input | 1(a), 1(b), 1(c) |
| 2 | A yes/no input | 2(a), 2(b) |
| 3 | More than one input | 3(a), 3(b) |
| 4 | Judging the classifier | 4(a), 4(b), 4(c), 4(d), 4(e), 4(f), 4(g) |
| 5 | A new dataset, end to end | 5(a), 5(b), 5(c), 5(d), 5(e) |

## Setup

This lab uses the same `CS_463` environment as the demos; there's nothing new to install.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from ISLP import load_data

plt.style.use('tableau-colorblind10')    # colors that stay distinct for color-blind readers

from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
                             classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

### The data: Default

The same ISL data as the demos: 10,000 simulated credit card customers.

| Column | Meaning |
|---|---|
| `default` | whether the customer defaulted on their debt: `No` or `Yes` |
| `student` | whether the customer is a student: `No` or `Yes` |
| `balance` | average monthly credit card balance, in dollars |
| `income` | annual income, in dollars |

The demos' helper file built a 0/1 copy of `default` for you. Here we build it ourselves.

In [ ]:
Default = load_data('Default')
Default['default01'] = (Default['default'] == 'Yes').astype(int)   # Yes -> 1, No -> 0
y = Default['default01'] # It is customary to use y to represent the target variable in code
Default.head()

In [ ]:
print(y.value_counts())
print(f'default rate: {y.mean():.1%}')

> **Positive means "defaulted."** As in NB2, the *positive* class is whatever is coded 1. In Parts 1–4 that's default. "Positive" isn't a judgment about whether the outcome is good.

## Part 1: A model with one input

Start with the model the demos used: default on standardized `balance`. You'll fit it with scikit-learn, read its parameters, and use it to score new customers.

<a id="how-fitting-works"></a>
### How fitting works

**Fitting is a search.** `fit` looks for the coefficients that make the outcomes in the data most likely: the *maximum likelihood* fit. There's no formula for them, so an **optimizer** searches. It starts from a guess and improves it a little at a time; each improvement is one **step** (scikit-learn calls it an *iteration*). It stops when a step barely helps anymore, or after `max_iter` steps (100 by default). If it hits `max_iter` first, scikit-learn prints a `ConvergenceWarning`: the coefficients may not have settled yet.

**`C=np.inf` turns off the penalty.** By default, scikit-learn doesn't fit plain maximum likelihood. It adds a **penalty** that charges the model for large coefficients, which pulls them toward 0. `C` sets how little the penalty counts: the default is `C=1`, a smaller `C` means a stronger penalty, and `C=np.inf` means no penalty at all. With `C=np.inf` you get the plain maximum likelihood fit that ISL and the demos describe, so this lab uses it whenever you'll interpret coefficients.

<details>
<summary>More detail: steps, the stopping rule, and the penalty</summary>

We're going to cover all of this in detail when we discuss gradient descent, but if you're curious, here are some details:

- **What a step does.** The optimizer measures how badly the current coefficients fit, using the *log-loss*: the average of $-\ln(\text{probability the model gave to what actually happened})$. Making the likelihood as large as possible is the same as making the log-loss as small as possible. Each step works like gradient descent: find the direction in which the log-loss drops fastest, and move the coefficients some distance that way. scikit-learn's default optimizer, `lbfgs`, also uses its recent steps to choose better directions and distances, so it usually needs only a few dozen steps.
- **The stopping rule.** `tol` (short for "tolerance", default `1e-4`) sets how small an improvement counts as "barely helps." `max_iter` is a safety limit. If you see a `ConvergenceWarning`, the usual first responses are to standardize the inputs and to raise `max_iter`. Part 5 shows a case where neither is the real fix.
- **The penalty.** With the default penalty, the optimizer minimizes the log-loss *plus* a cost that grows with the sum of the squared coefficients. `C` sets the balance between the two: the larger `C` is, the more the fit impacts loss and the less the penalty does.
- **Why the penalty is the default.** Shrinking coefficients toward 0 often makes predictions on *new* data better, and it keeps the fit well behaved when the data can't pin the coefficients down (Part 5). But shrunken coefficients won't match ISL's or the demos' estimates, which is why this lab turns it off. Chapter 6 covers penalties properly.

</details>

**1(a) · Code. (Medium Priority)** Standardize `balance` exactly as NB2 does, `(x - x.mean()) / x.std()`, and store it as a one-column DataFrame named `X_bal`. Fit `LogisticRegression(C=np.inf)` and name it `bal_model`. Store the intercept as `b0` and the slope as `b1`, and print both. You should get about $-6.06$ and $2.66$, matching NB2's fitted model.

<details>
<summary>Why standardize?</summary>

- **The problem: units.** Inputs come in whatever units they were measured in. Here, balance runs up to about \$2,650, income up to about \$74,000, and `student` is 0 or 1. Some methods treat a change of 1 in every input as comparable, so an input's units alone can make it count for more or less than it should.
- **What standardizing does.** Subtracting the mean and dividing by the standard deviation puts every input in the same unit: standard deviations from average. Now no input gets extra weight just from its units. As a bonus, coefficients become "per standard deviation," which makes them easier to compare (Part 3).
- **Plain logistic regression doesn't strictly need it.** The coefficients adapt to the units. Measure balance in cents instead of dollars, and its coefficient becomes 100 times smaller, with exactly the same predictions.
- **But the optimizer can struggle.** When inputs' scales differ a lot, a step size that suits one coefficient is far too big or too small for another, so the search takes more steps and can hit `max_iter` (see [How fitting works](#how-fitting-works)). Fitting raw balance and income takes the optimizer 50 steps here. After standardizing, it reaches the same model in 16.
- **Penalized logistic regression benefits regardless of the optimizer.** The penalty charges each coefficient for its size, and a coefficient's size depends on its input's units. Measure income in dollars and its coefficient is tiny, so the penalty barely touches it. Measure it in thousands of dollars and the same effect gets a coefficient 1,000 times larger, which the penalty does shrink. Standardizing makes the penalty treat every input the same way.

We'll discuss scale in class soon, including how to tell whether a model is likely to be sensitive to it.

</details>

`C=np.inf` turns off scikit-learn's default penalty, so you get the plain maximum likelihood fit. [How fitting works](#how-fitting-works), just above, explains what that means.

Also print two numbers you'll need in 1(b):

- one standard deviation of balance in dollars, since that's what "one unit" of standardized balance means;
- `expit(b0)`. SciPy's `expit` turns log-odds into a probability: it's the logistic function from NB1's *Solving for $p$*.

<details>
<summary>Hint 1: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
X_bal = ...        # standardized balance, as a one-column DataFrame
bal_model = ...
b0 = ...           # intercept
b1 = ...           # slope
# print b0, b1, one standard deviation of balance in dollars, and expit(b0)
```

</details>

<details>
<summary>Hint 2</summary>

Standardize the one-column DataFrame directly: `(Default[['balance']] - Default[['balance']].mean()) / Default[['balance']].std()`. The result is still a one-column DataFrame.

</details>

<details>
<summary>Think about: why does scikit-learn want <code>Default[['balance']]</code> (a DataFrame) rather than <code>Default['balance']</code> (a Series)?</summary>

scikit-learn treats `X` as a table: one row per customer and one column per input, even when there's only one input. A Series is 1-D, so it has no columns. Keeping `X` 2-D means the same code works unchanged when Part 3 adds more columns.

</details>

In [ ]:
# 1(a)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 1(a)
X_bal = (Default[['balance']] - Default[['balance']].mean()) / Default[['balance']].std()
bal_model = LogisticRegression(C=np.inf).fit(X_bal, y)
b0 = bal_model.intercept_[0]
b1 = bal_model.coef_[0, 0]

print(f'intercept b0 = {b0:.3f}, slope b1 = {b1:.3f} per standard deviation of balance')
print(f"one standard deviation of balance = ${Default['balance'].std():,.0f}")
print(f'expit(b0) = {expit(b0):.4f}')
```

</details>

**1(b) · Written.** Interpret `bal_model`'s two parameters, in dollars rather than standard deviations.

- **Slope:** complete this sentence: "Each additional \$___ of balance is associated with an increase of ___ in the log-odds of default."
- **Intercept:** a standardized balance of 0 is an average balance, about \$835. What are the log-odds of default for a customer with that balance, and what probability is that?

*Points back to:* NB1, *From clues to a model*: $\beta_0$ is the baseline evidence score, and $\beta_1 X$ is the evidence the input adds.

**1(b) Answer:**

_YOUR ANSWER HERE_


<details>
<summary>Our answer</summary>

Each additional \$484 of balance (one standard deviation) is associated with an increase of about 2.66 in the log-odds of default. (Equivalently, it multiplies the odds of default by about $e^{2.66} \approx 14$.) A customer with an average balance, about \$835, has log-odds of default of about $-6.06$: a probability of about 0.23%, so very unlikely to default. (We'll talk about how to do all of these calculations in class if you're not sure.)

</details>

**1(c) · Code. (High Priority)** There are four customers with standardized balances of: 0, 1, 2, and 3 (0 is an average balance). Use `bal_model.predict_proba` to find each one's probability of default, and show them in a DataFrame indexed by balance.

**What `predict_proba` does under the hood.** It runs the chapter's model forward, in two steps:

1. **Log-odds.** It computes the evidence score $z = \hat\beta_0 + \hat\beta_1 x$, which is the model's log-odds of default. You can get $z$ by itself from `bal_model.decision_function`.
2. **Probability.** It converts log-odds to a probability with the logistic function, $p = \dfrac{e^{z}}{1 + e^{z}}$, which is `expit(z)`. That's NB1's boxed result in *Solving for $p$*.

It returns one column per class, in the order of `bal_model.classes_`: $1 - p$ for class 0, then $p$ for class 1.

<details>
<summary>Hint: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
p_default = ...    # predict_proba's columns follow bal_model.classes_,
                   # so check which column is class 1, "default"
# show p_default in a DataFrame indexed by new_customers['balance']
```

</details>

In [ ]:
# 1(c)
new_customers = pd.DataFrame({'balance': [0, 1, 2, 3]})    # standardized balances
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 1(c)
new_customers = pd.DataFrame({'balance': [0, 1, 2, 3]})    # standardized balances
print('classes_:', bal_model.classes_)    # column 1 of predict_proba is class 1, "default"
p_default = bal_model.predict_proba(new_customers)[:, 1]

pd.DataFrame({'P(default)': p_default}, index=new_customers['balance']).round(4)
```

</details>

## Part 2: A yes/no input (about 5 minutes)

An input doesn't have to be a real number (or float) like balance. Here it's whether the customer is a `student`, coded 0/1.

**2(a) · Code.** Fit a logistic regression of `default01` on a 0/1 `student` column alone (1 for students), using `LogisticRegression(C=np.inf)`. Name it `student_model`, and print its intercept and coefficient.

Leave the 0/1 column unstandardized. As 0 and 1, its coefficient compares students with non-students.

<details>
<summary>Hint 1: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
X_student = ...        # one-column DataFrame: 1 for students, 0 for everyone else
student_model = ...
# print the intercept and the coefficient
```

</details>

<details>
<summary>Hint 2</summary>

scikit-learn wants `X` to be 2-D, even with one input. A one-column DataFrame works: `(Default[['student']] == 'Yes').astype(int)` (note the double brackets). `coef_` is 2-D too, so the slope is `coef_[0, 0]`.

</details>

In [ ]:
# 2(a)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 2(a)
X_student = (Default[['student']] == 'Yes').astype(int)
student_model = LogisticRegression(C=np.inf).fit(X_student, y)

print(f'intercept = {student_model.intercept_[0]:.3f}, '
      f'student coefficient = {student_model.coef_[0, 0]:.3f}')
```

</details>

**2(b) · Written.** Interpret the `student` coefficient in one sentence. On their own, do students look more or less likely to default than non-students, and by how much on the log-odds scale? Keep your answer in mind for Part 3.

**2(b) Answer:**

_YOUR ANSWER HERE_


<details>
<summary>Our answer</summary>

Students' log-odds of default are about 0.40 higher than non-students' (their odds are about 1.5 times as high), so on their own, students look *more* likely to default.

</details>

## Part 3: Multiple Logistic Regression (More than one input)

NB2's Section 11 used balance and income. Here you'll add `student` too. Read each coefficient the way you read multiple regression in the Ch 3 lab: *holding the other inputs fixed*.

**3(a) · Code.** Build `X_full` with three columns: standardized `balance`, standardized `income`, and a 0/1 `student` column. Leave `student` unstandardized, so its coefficient compares students with non-students. Fit `LogisticRegression(C=np.inf)` and name it `full_model`. Then show each input's coefficient in a table.

*Points back to:* NB2, Section 11. Its first code cell does almost exactly this, with two inputs.

<details>
<summary>Hint 1: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
X_full = ...                # balance and income, standardized
X_full['student'] = ...     # 0/1, not standardized
full_model = ...
# table: each input's coefficient
```

</details>

<details>
<summary>Hint 2</summary>

Standardize `Default[['balance', 'income']]` the same way you did `balance` alone (pandas standardizes each column separately). Then add the 0/1 column the same way you built it in 2(a): `X_full['student'] = (Default['student'] == 'Yes').astype(int)`.

</details>

In [ ]:
# 3(a)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 3(a)
X_full = Default[['balance', 'income']]
X_full = (X_full - X_full.mean()) / X_full.std()
X_full['student'] = (Default['student'] == 'Yes').astype(int)

full_model = LogisticRegression(C=np.inf).fit(X_full, y)
pd.Series(full_model.coef_[0], index=X_full.columns, name='coefficient').round(3)
```

</details>

One more fact for the next question: the average balance of students and non-students.

In [ ]:
Default.groupby('student', observed=True)['balance'].mean().round(0)

**3(b) · Written.** In 2(a), `student`'s coefficient was positive. In `full_model`, it's negative.

1. Interpret the new coefficient, using "holding balance and income fixed."
2. Explain why the sign flipped.
3. So, are students riskier customers or not? Say what each model's `student` coefficient would tell a lender.

<details>
<summary>Hint</summary>

Look at the average balances above, and at which input carries the most evidence in 3(a). Then remember the Ch 3 lab's `CompPrice` and `Price` example, where one predictor stood in for another it overlapped with.

</details>

**3(b) Answer:**

_YOUR ANSWER HERE_


<details>
<summary>Our answer</summary>

1. Holding balance and income fixed, a student's log-odds of default are about 0.65 *lower* than a non-student's. (Their odds are about half as high: $e^{-0.65} \approx 0.52$.)
2. Students carry higher balances on average (\$988 vs. \$772), and balance carries most of the evidence. Alone, `student` stood in for "probably has a higher balance," so students looked riskier. Once balance is in the model, it gets credit for its own evidence, and what's left of `student` points the other way.
3. Both coefficients are right; they answer different questions. If all a lender knows is whether someone is a student, the student is the riskier bet (2(a)). Comparing a student and a non-student with the *same* balance and income, the student is the safer bet (3(a)). Neither coefficient says being a student *causes* anything. Each describes the data, given which other inputs are in the model.

</details>

## Part 4: Judging the classifier (about 25 minutes)

Back to `bal_model`, the balance-only model, so your numbers match NB2. First, every customer's predicted probability of default:

In [ ]:
p_bal = bal_model.predict_proba(X_bal)[:, 1]    # column 1 = class 1, "default"

> **Caution:** we're evaluating the model against the same customers it was fit to. Real evaluation uses data the model never saw during fitting. Chapter 5 covers how to do that.

**4(a) · Code.** Turn `p_bal` into 0/1 predictions at threshold 0.5 and name them `pred_50`. Build the confusion matrix with `confusion_matrix(y, pred_50)`, name it `cm`, and unpack its four counts into `tn`, `fp`, `fn`, and `tp`. Print the matrix and the four counts.

To unpack the counts, use `tn, fp, fn, tp = cm.ravel()`, the usual idiom (it's in scikit-learn's `confusion_matrix` documentation). `ravel()` flattens an array into one row, reading the matrix row by row. So `[[TN, FP], [FN, TP]]` becomes `TN, FP, FN, TP`, in that order. Check the four counts against the printed matrix.

*Points back to:* NB2, Section 4. Rows are what actually happened and columns are what the model predicted, scikit-learn's layout. ISL's tables are the other way around.

<details>
<summary>Hint 1: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
pred_50 = ...           # 0/1 predictions at threshold 0.5
cm = ...
tn, fp, fn, tp = ...    # the four counts, from cm
# print cm and the four counts
```

</details>

<details>
<summary>Hint 2</summary>

`(p_bal >= 0.5).astype(int)` makes the predictions, as in NB2.

</details>

In [ ]:
# 4(a)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 4(a)
pred_50 = (p_bal >= 0.5).astype(int)
cm = confusion_matrix(y, pred_50)
tn, fp, fn, tp = cm.ravel()

print(cm)
print(f'TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}')
```

</details>

Run this cell to draw your matrix. *(You don't need to understand how this plot was generated.)*

In [ ]:
disp = ConfusionMatrixDisplay(cm, display_labels=['no default', 'default'])
disp.plot(cmap='Blues', colorbar=False)
disp.ax_.set(xlabel='Predicted', ylabel='Actual', title='Balance model at threshold 0.5')
plt.show()

**4(b) · Code. (The by-hand part: Low Priority)** Using only `tn`, `fp`, `fn`, and `tp`, compute accuracy, recall, precision, specificity, and the false positive rate. Then check the first three against scikit-learn's `accuracy_score`, `recall_score`, and `precision_score`, and show both versions side by side. scikit-learn has no functions for specificity or the false positive rate; computing them from the confusion matrix, as you just did, is the usual way.

*Points back to:* NB2, Section 5's table. Write the formulas from that table yourself.

<details>
<summary>Hint: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
by_hand = {
    'accuracy': ...,
    'recall': ...,
    'precision': ...,
    'specificity': ...,
    'false positive rate': ...,
}
sklearn_versions = {
    'accuracy': ...,
    'recall': ...,
    'precision': ...,
}
# pd.DataFrame({'by hand': by_hand, 'scikit-learn': sklearn_versions}) lines the two up
# by key, with NaN where scikit-learn has no version
```

</details>

In [ ]:
# 4(b)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 4(b)
by_hand = {
    'accuracy': (tp + tn) / (tp + tn + fp + fn),
    'recall': tp / (tp + fn),
    'precision': tp / (tp + fp),
    'specificity': tn / (tn + fp),
    'false positive rate': fp / (tn + fp),
}
sklearn_versions = {
    'accuracy': accuracy_score(y, pred_50),
    'recall': recall_score(y, pred_50),
    'precision': precision_score(y, pred_50),
}
pd.DataFrame({'by hand': by_hand, 'scikit-learn': sklearn_versions}).round(4)
```

</details>

**4(c) · Code + Written.** Fit `DummyClassifier(strategy='most_frequent')` on `X_bal` and `y`. It ignores its inputs and always predicts the most common class, just like NB2's "always no" classifier. Compare its accuracy, recall, and precision with `pred_50`'s. Then answer: which metric would have hidden the difference between the two classifiers, and why?

*Points back to:* NB2, Section 2.

<details>
<summary>Hint: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
dummy = ...         # fit on X_bal and y
pred_dummy = ...
# table: accuracy, recall, and precision for both classifiers.
# The dummy never flags anyone, so its precision is 0/0:
# pass zero_division=0 to precision_score to skip the warning.
```

</details>

In [ ]:
# 4(c)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 4(c)
dummy = DummyClassifier(strategy='most_frequent').fit(X_bal, y)
pred_dummy = dummy.predict(X_bal)

pd.DataFrame({
    'always "no"': [accuracy_score(y, pred_dummy), recall_score(y, pred_dummy),
                    precision_score(y, pred_dummy, zero_division=0)],
    'balance model': [accuracy_score(y, pred_50), recall_score(y, pred_50),
                      precision_score(y, pred_50)],
}, index=['accuracy', 'recall', 'precision']).round(4)
```

</details>

**4(c) Answer:**

_YOUR ANSWER HERE_


<details>
<summary>Our answer</summary>

Accuracy. The two classifiers' accuracies are almost the same (96.67% vs. 97.25%), but the dummy catches no defaulters (recall 0) while the model catches 30% of them. Only 3.3% of customers default, so the 9,667 non-defaulters dominate accuracy, and saying "no" to everyone is right almost all the time.

</details>

**`classification_report`** prints these metrics for each class at once. Run the cell below.

- The `default` row shows the recall and precision you computed.
- The `no default` row treats *non*-default as the positive class, so its "recall" is your specificity.
- `support` is the number of customers actually in each class.

In [ ]:
print(classification_report(y, pred_50, target_names=['no default', 'default'], digits=3))

**4(d) · Code.** Write a function `metrics_by_threshold(y_true, p, thresholds)`. It should return a DataFrame with one row per threshold and these columns: `threshold`, `recall`, `precision`, `f1`, and `false alarms` (the number of false positives). Then run it on the balance model and name the result `sweep`. Write it as a function, because you'll reuse it in Part 5.

*Points back to:* NB2, Demos 6.2 and 7.2.

<details>
<summary>Hint: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
def metrics_by_threshold(y_true, p, thresholds):
    y_true = np.asarray(y_true)
    rows = []
    for t in thresholds:
        pred = ...    # 0/1 predictions at threshold t
        rows.append({'threshold': t,
                     'recall': ...,
                     'precision': ...,    # zero_division=np.nan: when nobody is flagged,
                                          # precision is 0/0, undefined rather than 0
                     'f1': ...,
                     'false alarms': ...})
    return pd.DataFrame(rows)


sweep = ...
```

</details>

In [ ]:
# 4(d)
thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 4(d)
thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
def metrics_by_threshold(y_true, p, thresholds):
    y_true = np.asarray(y_true)
    rows = []
    for t in thresholds:
        pred = (p >= t).astype(int)
        rows.append({'threshold': t,
                     'recall': recall_score(y_true, pred),
                     'precision': precision_score(y_true, pred, zero_division=np.nan),
                     'f1': f1_score(y_true, pred),
                     'false alarms': int(((pred == 1) & (y_true == 0)).sum())})
    return pd.DataFrame(rows)


sweep = metrics_by_threshold(y, p_bal, thresholds)
sweep.head()
```

</details>

Run these cells to plot your `sweep`. *(You don't need to understand how this plot was generated.)*

In [ ]:
def plot_threshold_sweep(sweep, title):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(sweep['threshold'], sweep['recall'], label='recall', linestyle='-')
    ax.plot(sweep['threshold'], sweep['precision'], label='precision', linestyle='-.')
    ax.plot(sweep['threshold'], sweep['f1'], label='F1', linestyle='--')
    ax.set(xlabel='threshold', ylabel='metric value', title=title, xlim=(0, 1), ylim=(0, 1.02))
    ax.legend()
    plt.show()

In [ ]:
plot_threshold_sweep(sweep, 'Balance model: recall, precision, and F1 at every threshold')

**4(e) · Code + Written.** The bank's risk team says: *"Catch at least 75% of defaulters."*

1. Use `sweep` to find the **highest** threshold that meets the requirement. Name it `t_75`.
2. Find the threshold with the highest F1. Name it `t_f1`.
3. Show the `sweep` rows for `t_75`, `t_f1`, and 0.5.

Then answer: which of the three thresholds would you recommend to the bank, and why? Which kind of mistake is the risk team's requirement most worried about?

*Points back to:* NB2, Section 3 (*which mistake costs the bank more?*) and the end of Section 6: "0.5 is a choice, not a law."

<details>
<summary>Hint 1: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
t_75 = ...    # highest threshold with recall of at least 0.75
t_f1 = ...    # threshold with the best F1
# show the sweep rows for t_75, t_f1, and 0.5
```

</details>

<details>
<summary>Hint 2</summary>

Filter with `sweep[sweep['recall'] >= 0.75]`, then take the largest `threshold`. Why the *highest* qualifying threshold? Lowering it further would still meet the requirement, but it would add false alarms. `sweep['f1'].idxmax()` gives the row label of the best F1.

</details>

In [ ]:
# 4(e)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 4(e)
t_75 = sweep.loc[sweep['recall'] >= 0.75, 'threshold'].max()
t_f1 = sweep.loc[sweep['f1'].idxmax(), 'threshold']

print(f'highest threshold with recall >= 0.75: {t_75}')
print(f'threshold with the best F1:            {t_f1}')
sweep[sweep['threshold'].isin([t_75, t_f1, 0.5])].set_index('threshold').round(3)
```

</details>

**4(e) Answer:**

_YOUR ANSWER HERE_


<details>
<summary>Our answer</summary>

The requirement is about **false negatives**, the missed defaulters: the bank cares most about not letting them through. Only 0.08 meets it. It catches 78% of defaulters, at the price of 673 false alarms, so only about 28% of flagged customers actually default. The F1 threshold (0.32) catches only 49%, and 0.5 catches only 30%, so neither meets the requirement. F1 also counts a missed defaulter and a false alarm as equally bad, which isn't how the risk team sees it. A threshold of 0.08 is fine as long as the bank can afford to review about 930 flagged customers.

</details>

**4(f) · Code.** Compare three models by AUC:

- `bal_model` (balance only), using `p_bal`;
- a new model on standardized income only: name it `income_model`, and its probabilities `p_income`;
- `full_model` from Part 3: name its probabilities `p_full`.

Compute each model's AUC with `roc_auc_score(y, probabilities)`.

*Points back to:* NB2, Demo 8.2.

<details>
<summary>Hint: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
X_income = ...        # standardized income, as a one-column DataFrame
income_model = ...
p_income = ...
p_full = ...          # full_model's probabilities (it takes X_full)
# each model's AUC, in a Series
```

</details>

In [ ]:
# 4(f)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 4(f)
X_income = (Default[['income']] - Default[['income']].mean()) / Default[['income']].std()
income_model = LogisticRegression(C=np.inf).fit(X_income, y)
p_income = income_model.predict_proba(X_income)[:, 1]
p_full = full_model.predict_proba(X_full)[:, 1]

pd.Series({'balance only': roc_auc_score(y, p_bal),
           'income only': roc_auc_score(y, p_income),
           'balance + income + student': roc_auc_score(y, p_full)}, name='AUC').round(4)
```

</details>

Run this cell to draw the three ROC curves. *(You don't need to understand how this plot was generated.)*

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name, p, style in [('balance only', p_bal, '-'), ('income only', p_income, '-.'),
                       ('balance + income + student', p_full, '--')]:
    RocCurveDisplay.from_predictions(y, p, name=name, ax=ax, curve_kwargs={'linestyle': style})
ax.plot([0, 1], [0, 1], 'k:', label='random guessing')
ax.set(xlabel='false positive rate', ylabel='recall (true positive rate)',
       title='ROC curves for three Default models')
ax.legend(loc='lower right')
plt.show()

**4(g) · Written.** Two short questions:

1. Adding income and student status barely moved the AUC (0.948 to 0.950). Why, given the coefficients in 3(a)?
2. If the bank moves its threshold from 0.5 to 0.08, does the balance model's AUC change?

*Points back to:* NB2, Section 8: what AUC measures.

**4(g) Answer:**

_YOUR ANSWER HERE_


<details>
<summary>Our answer</summary>

1. Look at the ROC plot: the two curves nearly coincide. Income and student status add little evidence beyond what balance already provides. Income's coefficient is tiny (0.04 per standard deviation), and student's ($-0.65$) is small next to balance's 2.78 per standard deviation. So at any false positive rate, the full model catches about the same share of defaulters as the balance model. Nearly the same tradeoff everywhere means nearly the same curve, and so nearly the same area.
2. No. AUC summarizes the whole ROC curve, every threshold at once. Changing the threshold the bank chooses to deploy/use doesn't change a curve based on *every* threshold.

</details>

## Part 5: A new dataset, end to end (about 20 minutes)

Same workflow, new problem, less scaffolding. scikit-learn ships a real medical dataset: 569 breast tumors, each described by 30 measurements of cell nuclei in a biopsy image, and labeled malignant (cancer) or benign.

**5(a) · Code + Written.** Load the data with `load_breast_cancer(as_frame=True)` and name it `cancer`. Print `cancer.target_names` and `cancer.target.value_counts()`. Then build:

- `X_cancer`: the 30 measurements (`cancer.data`);
- `y_cancer`: a 0/1 target where **1 means malignant**, so that "positive" means cancer.

Then answer: if you'd used `cancer.target` as it comes, what would "recall" have measured?

*Points back to:* the *"Positive means..."* callout in NB2 (and in this lab's Setup).

<details>
<summary>Hint 1: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
cancer = ...
# print cancer.target_names and cancer.target.value_counts()
X_cancer = ...
y_cancer = ...    # 1 = malignant, 0 = benign
```

</details>

<details>
<summary>Hint 2</summary>

`target_names[k]` is the name of class `k`. Look at which name is class 0.

</details>

In [ ]:
# 5(a)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 5(a)
cancer = load_breast_cancer(as_frame=True)
print(cancer.target_names)          # class 0 = malignant, class 1 = benign
print(cancer.target.value_counts())

X_cancer = cancer.data
y_cancer = (cancer.target == 0).astype(int)    # 1 = malignant (positive), 0 = benign
print(y_cancer.value_counts())
```

</details>

**5(a) Answer:**

_YOUR ANSWER HERE_


<details>
<summary>Our answer</summary>

In scikit-learn's coding, 0 is malignant and 1 is benign, so the positive class would have been *benign*. Recall would have measured the share of benign tumors the model found, which tells the clinic nothing about how many cancers it catches. A "recall ≥ 0.95" requirement would have been met by a model that misses plenty of cancers.

</details>

**Pipelines.** In Parts 1 and 3 you standardized by hand. In practice, you'd usually chain the scaler and the model into a **pipeline**:

```python
make_pipeline(StandardScaler(), LogisticRegression(C=np.inf))
```

- `fit` learns each column's mean and standard deviation, standardizes, and then fits the model.
- `predict_proba` applies the *same* standardization before predicting.

The scaling travels with the model, so you can't forget it when new data arrives. `model[-1]` is the fitted `LogisticRegression` inside, if you want its coefficients.

**5(b) · Code.** Fit a pipeline on two inputs, `'worst radius'` and `'worst concave points'`. These are the largest ("worst") values in each image for the size of a nucleus and for the number of dents in its outline. Name the model `cancer_model` and its predicted probabilities of malignancy `p_cancer`. At threshold 0.5, report the confusion matrix, recall, precision, and AUC.

<details>
<summary>Hint: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
cancer_model = ...    # the pipeline, fit on X_cancer[two_inputs] and y_cancer
p_cancer = ...        # predicted probabilities of malignancy
pred_cancer = ...     # 0/1 at threshold 0.5
# print the confusion matrix, recall, precision, and AUC
```

</details>

In [ ]:
# 5(b)
two_inputs = ['worst radius', 'worst concave points']
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 5(b)
two_inputs = ['worst radius', 'worst concave points']
cancer_model = make_pipeline(StandardScaler(), LogisticRegression(C=np.inf))
cancer_model.fit(X_cancer[two_inputs], y_cancer)
p_cancer = cancer_model.predict_proba(X_cancer[two_inputs])[:, 1]
pred_cancer = (p_cancer >= 0.5).astype(int)

print(confusion_matrix(y_cancer, pred_cancer))
print(f'recall    {recall_score(y_cancer, pred_cancer):.3f}')
print(f'precision {precision_score(y_cancer, pred_cancer):.3f}')
print(f'AUC       {roc_auc_score(y_cancer, p_cancer):.3f}')
```

</details>

**5(c) · Code + Written.** A screening clinic requires recall of at least 0.95. Use your `metrics_by_threshold` from 4(d) to find the highest threshold that meets it, and name it `t_clinic`. Show the sweep rows for `t_clinic` and 0.5, and plot the sweep with `plot_threshold_sweep`. How many benign tumors get flagged at `t_clinic`?

Then answer: a spam filter's team would probably refuse a trade like this. Why is it reasonable for the clinic?

*Points back to:* NB2, Section 3's *Ask* about the hospital and the spam filter.

<details>
<summary>Hint: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
cancer_sweep = ...    # metrics_by_threshold from 4(d)
t_clinic = ...        # highest threshold with recall of at least 0.95
# show the sweep rows for t_clinic and 0.5, then plot with plot_threshold_sweep
```

</details>

In [ ]:
# 5(c)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 5(c)
cancer_sweep = metrics_by_threshold(y_cancer, p_cancer, thresholds)
t_clinic = cancer_sweep.loc[cancer_sweep['recall'] >= 0.95, 'threshold'].max()

print(f'highest threshold with recall >= 0.95: {t_clinic}')
display(cancer_sweep[cancer_sweep['threshold'].isin([t_clinic, 0.5])].set_index('threshold').round(3))
plot_threshold_sweep(cancer_sweep, 'Breast cancer model: recall, precision, and F1 at every threshold')
```

</details>

**5(c) Answer:**

_YOUR ANSWER HERE_


<details>
<summary>Our answer</summary>

At 0.37, the model catches 95.3% of malignant tumors and flags 21 of the 357 benign ones. For the clinic, a missed cancer (false negative) can be deadly, while a false alarm (false positive) means a follow-up test and some worry, so trading false alarms for caught cancers is worth it. For a spam filter, a false positive (a real email lost to the spam folder) is the costlier mistake, so the team would push the threshold the other way.

</details>

**5(d) · Code + Written.** Now use all 30 inputs, unpenalized: `make_pipeline(StandardScaler(), LogisticRegression(C=np.inf))`. Fit it twice:

- `all30_default`: with the default `max_iter`, the number of steps the optimizer may take (100; see [How fitting works](#how-fitting-works) in Part 1);
- `all30_long`: with `LogisticRegression(C=np.inf, max_iter=10_000)`.

For each fit, report training accuracy, the largest absolute coefficient, and how many predicted probabilities are within $10^{-6}$ of 0 or 1. Watch the cell's output for warnings, too.

Then answer:

1. **Warning signs.** When the two classes are perfectly separated (some boundary puts every malignant tumor on one side and every benign one on the other), the maximum likelihood fit doesn't exist. The optimizer can always improve the fit by making the coefficients bigger, so it stops wherever its step limit or stopping rule says to. Fits like that usually show some of these signs:
   - huge coefficients;
   - coefficients that keep growing when the optimizer is allowed more steps;
   - a convergence warning;
   - predicted probabilities at, or almost at, exactly 0 or 1.

   Which of these do you see?
2. **Causes.** Separation is most likely when there's (a) a small sample, especially with a rare class, so a few positives are easy to fence off; (b) many inputs, since each one adds a direction to split along; or (c) one input that gives the answer away, like a "sent to collections" flag when predicting default. Which applies here?
3. Should you believe the 100% accuracy?

*For more:* NB2, Section 10, especially *What software does about it*.

<details>
<summary>Hint: code skeleton</summary>

Paste this into the code cell and replace each `...`.

```python
def separation_report(model, X, y_true):
    p = ...    # predicted probabilities of malignancy
    return {'training accuracy': ...,
            'largest |coefficient|': ...,    # the coefficients are in model[-1].coef_
            'probabilities within 1e-6 of 0 or 1': ...}


all30_default = ...    # default max_iter
all30_long = ...       # max_iter=10_000
# show both reports side by side in a DataFrame
```

</details>

In [ ]:
# 5(d)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 5(d)
def separation_report(model, X, y_true):
    p = model.predict_proba(X)[:, 1]
    return {'training accuracy': accuracy_score(y_true, model.predict(X)),
            'largest |coefficient|': np.abs(model[-1].coef_).max(),
            'probabilities within 1e-6 of 0 or 1': int(((p < 1e-6) | (p > 1 - 1e-6)).sum())}


all30_default = make_pipeline(StandardScaler(), LogisticRegression(C=np.inf)).fit(X_cancer, y_cancer)
all30_long = make_pipeline(StandardScaler(), LogisticRegression(C=np.inf, max_iter=10_000)).fit(X_cancer, y_cancer)

pd.DataFrame({'default max_iter': separation_report(all30_default, X_cancer, y_cancer),
              'max_iter=10,000': separation_report(all30_long, X_cancer, y_cancer)}).round(3)
```

</details>

**5(d) Answer:**

_YOUR ANSWER HERE_


<details>
<summary>Our answer</summary>

1. All four. The coefficients are huge, and they grow with more steps: about 24 when it gave up after the default number of steps (with a `ConvergenceWarning`), and about 7,900 when it was allowed 10,000 steps (with no warning at all). And most predicted probabilities are pinned at 0 or 1: 557 of 569 in the longer fit. The classes are perfectly separated, so the maximum likelihood estimate doesn't exist, and the "estimate" is just wherever the optimizer quit.
2. **(b), many inputs**: 30 measurements for only 569 tumors gives the model plenty of directions in which to fence off the malignant ones.
3. No. It's accuracy on the same tumors the model was fit to, and separating these particular 569 tumors perfectly says little about the next one. The probabilities of exactly 0 and 1 are far more confident than the evidence justifies.

</details>

**5(e) · Code.** Refit all 30 inputs with scikit-learn's **default penalty** ([How fitting works](#how-fitting-works)), `LogisticRegression()`, and name the model `all30_penalized`. Report the same three numbers with your code from 5(d).

In [ ]:
# 5(e)
### YOUR CODE HERE


<details>
<summary>Solution</summary>

```python
# 5(e)
all30_penalized = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_cancer, y_cancer)
pd.Series(separation_report(all30_penalized, X_cancer, y_cancer)).round(3)
```

</details>

The penalty charges the model for large coefficients, which restores the tension the separated data lost (NB2, Section 10). The largest coefficient drops from thousands to about 1.3. Some probabilities are still extreme: 30 modest coefficients can add up to an overwhelming evidence score. Chapter 6 covers penalties properly, and Chapter 5 covers how to measure accuracy honestly, on tumors the model hasn't seen.

## Wrapping up

- **Reading coefficients (Parts 1–3):** a coefficient is the change in log-odds for a one-unit step in its input, *holding the other inputs in the model fixed*. Which inputs are in the model changes what a coefficient means, and can even flip its sign: on their own, students look riskier, but a student is less risky than a non-student with the same balance and income.
- **Using a fitted model:** `predict_proba` gives a probability for each class, in the order of `classes_`.
- **Judging a classifier (Part 4):** accuracy can mislead us about a useless classifier when one class is rare. The confusion matrix, recall, precision, and F1 show what's really happening, and the threshold is a choice you make from what the problem needs. AUC sums up the whole ROC curve, every threshold at once, in one number.
- **New data (Part 5):** check which class is coded positive before trusting any metric. Huge coefficients and probabilities of exactly 0 or 1 signal separation, and a penalty is the usual fix.
- **Still to come:** everything here was measured on the data the model was fit to. Chapter 5 shows how to evaluate a model on data it hasn't seen.